# EchoAI — Qwen2.5-3B-Instruct fine-tune · Step 3: Training

Loads Qwen2.5-3B-Instruct in bf16 (no bnb, no quantization), applies high-rank LoRA, trains 3 epochs on `dataset_v1_train.jsonl`, evals on 10 val_prompts after each epoch, saves merged FP16 model to Drive.

**Why bf16 + LoRA (no 4-bit)?** Avoids the bitsandbytes/triton mess. Qwen2.5-3B bf16 = ~6 GB. With LoRA + grad checkpoint, total VRAM ~9 GB on T4 (16 GB). Comfortable.

**Aggressive-forgetting recipe:** r=64, alpha=128, ALL 7 linear modules, dropout=0, lr=2e-4 cosine, 3 epochs.

**Expected runtime:** ~2 hours on T4.

## Cell 1 — Setup

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU! Switch runtime to T4.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/qwen_ft'
DRIVE_DIR = '/content/drive/MyDrive/echoai_qwen_ft'
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print('Workspace :', WORK)
print('Drive dir :', DRIVE_DIR)

In [ ]:
!pip install -q -U "transformers>=4.48.0" "accelerate>=1.2.0" "peft>=0.14.0" "datasets>=3.1.0"
print('\nInstalled. NO restart needed (no bnb).')

## Cell 2 — Pull datasets from Drive

In [ ]:
import shutil
for f in ['dataset_v1_train.jsonl','dataset_v1_val.jsonl','val_prompts.jsonl','heldout_prompts.jsonl']:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{WORK}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        n = sum(1 for _ in open(dst))
        print(f'  {f}: {n} lines')
    else:
        print(f'  MISSING: {src}')

## Cell 3 — Load Qwen2.5-3B-Instruct in bf16

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'right'  # for training; we'll flip to left for generation later

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
print('Base loaded:', BASE_MODEL)
print('Mem (GB) :', round(torch.cuda.memory_allocated()/1e9, 2))

## Cell 4 — Apply LoRA (high rank, all 7 modules)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_cfg = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.0,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model.enable_input_require_grads()  # required when base is frozen + grad_ckpt
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('Mem (GB) after LoRA:', round(torch.cuda.memory_allocated()/1e9, 2))

## Cell 5 — Tokenize dataset with Qwen ChatML template

In [ ]:
import json
from datasets import Dataset

MAX_LEN = 1024  # most samples are well under 512; cap at 1024 to keep T4 happy

def load_jsonl(path):
    return [json.loads(l) for l in open(path)]

train_raw = load_jsonl(f'{WORK}/dataset_v1_train.jsonl')
val_raw   = load_jsonl(f'{WORK}/dataset_v1_val.jsonl')
print(f'train={len(train_raw)} val={len(val_raw)}')

def to_text(sample):
    """Apply Qwen ChatML template; we train on the full sequence (loss on all tokens including user — 
    standard SFT shortcut; for stricter loss masking we'd need a custom collator)."""
    return tok.apply_chat_template(sample['messages'], tokenize=False, add_generation_prompt=False)

def tokenize_fn(batch):
    texts = [to_text({'messages': m}) for m in batch['messages']]
    out = tok(texts, truncation=True, max_length=MAX_LEN, padding=False)
    out['labels'] = [ids.copy() for ids in out['input_ids']]
    return out

train_ds = Dataset.from_list(train_raw).map(tokenize_fn, batched=True, batch_size=64, remove_columns=['messages'])
val_ds   = Dataset.from_list(val_raw  ).map(tokenize_fn, batched=True, batch_size=64, remove_columns=['messages'])
print('Tokenized. Sample lens:', sorted(len(x) for x in train_ds[:5]['input_ids']))

## Cell 6 — Trainer setup

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUT_DIR = f'{WORK}/checkpoints'

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,        # effective batch 16
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,
    optim='adamw_torch',
    report_to='none',
    remove_unused_columns=False,
    label_names=['labels'],
    dataloader_num_workers=2,
)

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    tokenizer=tok,
)
print('Trainer ready. Steps/epoch:', len(train_ds) // 16)
print('Total steps:', (len(train_ds) // 16) * 3)

## Cell 7 — Custom callback: generate on val_prompts after each epoch

After each epoch, generate answers to the 10 in-training val_prompts. Logs citation-presence and answer-length — quick proxy for whether the model is learning the format.

In [ ]:
import re
from transformers import TrainerCallback

val_prompts = load_jsonl(f'{WORK}/val_prompts.jsonl')
SYSTEM_PROMPT = (
    'You are an expert on the Constitution of the Islamic Republic of Pakistan. '
    'Answer using only the text of the Constitution. Quote the relevant clause verbatim '
    'and end every answer with the citation in square brackets, e.g. [Art 25(1)] or '
    "[Art 260(1)]. If the question is not addressed in the Constitution, reply with: "
    "'This is not addressed in the Constitution of Pakistan. [no citation]'"
)
CITE_RE = re.compile(r'\[(Art|Part|Chapter|Preamble|no citation)')

@torch.inference_mode()
def generate_one(model, tok, user_msg, max_new_tokens=384):
    msgs = [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':user_msg}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    model.eval()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    model.train()
    return tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

class GenEvalCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        model_ref = kwargs['model']
        n_cited = 0; total_chars = 0
        print(f'\n========== End of epoch {int(state.epoch)} — val_prompts eval ==========')
        for p in val_prompts:
            ans = generate_one(model_ref, tok, p['prompt'])
            cited = bool(CITE_RE.search(ans))
            n_cited += int(cited); total_chars += len(ans)
            marker = '✓' if cited else '✗'
            print(f"  [{p['tier']}] {marker} Q{p['id']}: {ans[:140].replace(chr(10),' ')}...")
        print(f'  → citation rate: {n_cited}/{len(val_prompts)}  ·  avg answer: {total_chars/len(val_prompts):.0f} chars')

trainer.add_callback(GenEvalCallback())
print('Callback registered.')

## Cell 8 — TRAIN (~2 hours on T4)

Resume-safe: if T4 disconnects, re-run from Cell 1 → 6 → 8 with `resume_from_checkpoint=True`.

In [ ]:
result = trainer.train()
print('\nTraining complete.')
print('Final loss:', result.training_loss)

## Cell 9 — Save LoRA adapter + merged FP16 model to Drive

In [ ]:
# Save LoRA adapter (small, ~250 MB)
adapter_path = f'{DRIVE_DIR}/qwen25_3b_constitution_lora'
trainer.model.save_pretrained(adapter_path)
tok.save_pretrained(adapter_path)
print('[✓] LoRA adapter →', adapter_path)

# Free VRAM before merge
del trainer
torch.cuda.empty_cache()

# Merge adapter into base for deployment (~6 GB FP16)
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto')
merged = PeftModel.from_pretrained(base, adapter_path)
merged = merged.merge_and_unload()

merged_path = f'{DRIVE_DIR}/qwen25_3b_constitution_v1'
merged.save_pretrained(merged_path, safe_serialization=True)
tok.save_pretrained(merged_path)
print('[✓] Merged FP16 model →', merged_path)

## Cell 10 — Final held-out eval (10 prompts NEVER seen in training)

In [ ]:
held = load_jsonl(f'{WORK}/heldout_prompts.jsonl')
results = []
n_cited = 0; n_refuse = 0
print('===== Held-out eval (Q never seen in training) =====\n')
for p in held:
    ans = generate_one(merged, tok, p['prompt'])
    cited = bool(re.search(r'\[(Art|Part|Chapter|Preamble)', ans))
    refused = '[no citation]' in ans
    n_cited += int(cited); n_refuse += int(refused)
    results.append({'id':p['id'], 'tier':p['tier'], 'q':p['prompt'], 'a':ans, 'cited':cited, 'refused':refused})
    print(f"--- Q{p['id']} [{p['tier']}] {'✓ CITED' if cited else ('REFUSED' if refused else '✗ NO-CITE')} ---")
    print(f"Q: {p['prompt']}")
    print(f"A: {ans[:500]}")
    print()

print(f'\n[summary] cited={n_cited}/10  refusals={n_refuse}/10  no-cite={10-n_cited-n_refuse}/10')

# Save eval report to Drive
with open(f'{DRIVE_DIR}/heldout_eval_report.json','w') as f:
    json.dump({'results':results, 'cited':n_cited, 'refused':n_refuse}, f, indent=2)
print('[✓] Report →', f'{DRIVE_DIR}/heldout_eval_report.json')

## Done.

**Deliverables in Drive `echoai_qwen_ft/`:**
- `qwen25_3b_constitution_lora/` — LoRA adapter (~250 MB)
- `qwen25_3b_constitution_v1/` — Merged FP16 model (~6 GB), ready to load with `AutoModelForCausalLM.from_pretrained()`
- `heldout_eval_report.json` — final eval results

**To use the model anywhere:**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained('path/to/qwen25_3b_constitution_v1', torch_dtype='bfloat16', device_map='auto')
tok = AutoTokenizer.from_pretrained('path/to/qwen25_3b_constitution_v1')
```